In [ ]:
import os
import sys

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations

from abyss.glob import glob
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def rename_duplicates(df):
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        print(dup)
        cols[cols[cols == dup].index.values.tolist()] = [dup + '_' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
    df.columns = cols
    return df

In [ ]:
phenotypes_labels = {
    "bc": "Breast Cancer",
    "cad": "Coronary Artery Disease",
    "ms": "Multiple Sclerosis",
    "t2d": "Type 2 Diabetes",
    "pc": "Prostate Cancer",
    "ibd": "Inflammatory Bowel Disease"
}

scenario_labels = {
    "no_cov": "no covariates",
    "abyss": "Abyss correction",
    "cov": "PCA correction",
    "abyss_cov": "Abyss + PCA as covariates"
}

In [ ]:
# Parse command line arguments
arguments = {
    'bin':17,
    'pheno':'ms',
    'PCs':5,
    'covs':"pca",
    'scenario':"abyss_cov",
    'bottle_latent':5
}

# Set default values if arguments are not provided
bin = arguments['bin']
pheno_name = arguments['pheno']

try:
    PCs = int(arguments['PCs'])
except ValueError:
    PCs = 5

try:
    latent_dim = int(arguments['bottle_latent'])
except ValueError:
    latent_dim = 5

try:
    covname = arguments['covs']
    path_cov = f"{glob.PATH_COVS}/{covname}.pkl"
    covs = pd.read_pickle(path_cov)
    print(covs)
except:
    pass

# Possibility to choose between 4 scenarios: "cov" "abyss" "abyss_cov" "no_cov"
try:
    scenario = arguments['scenario']
except:
    scenario = "abyss"


In [ ]:
pheno = pd.read_pickle(f"{glob.PATH_PHENO}/{pheno_name}.pkl")

In [ ]:
path_dims = f"{glob.PATH_GENO}/07_lowest_dimensionality/bin_{bin}/{latent_dim}_dim_latent/"
existing_dims = [f for f in os.listdir(path_dims) if f.startswith("reduction")]
try:
    pheno_name = arguments['pheno']
    accu = pd.read_pickle(f"{glob.PATH_GENO}/07_lowest_dimensionality/bin_{bin}/{pheno_name}_best_bottles.pkl")
    dims = int(accu.sort_values(by='accuracy', ascending=False)[['bottleneck']].iloc[0, 0])
except Exception as e:
    print(e)
    path_dims = f"{glob.PATH_GENO}/07_lowest_dimensionality/bin_{bin}/{latent_dim}_dim_latent/"
    dims = int(existing_dims[-1].split("_")[3])
dims = 6

In [ ]:
dims

In [ ]:
path_output_GWAS =  glob.PATH_GENO + f"/09_summary_statistics/{pheno_name}/"
os.makedirs(path_output_GWAS, exist_ok=True)

path_output = path_output_GWAS + scenario
os.makedirs(path_output, exist_ok=True)
path_input_maf_prob = glob.PATH_GENO + f"/08_MAF_probability_landscape/{pheno_name}/bin_{bin}/{dims}_bottleneck_dims/"
path_input_geno = glob.PATH_GENO + f"/04_LD_blocks/{PCs}_PCs/bin_{bin}/"
input_maf_prob = [f for f in os.listdir(path_input_maf_prob) if not f.startswith("bin")]
input_geno = [f for f in os.listdir(path_input_geno) if not f.startswith("mapping")]

In [ ]:
if scenario == 'abyss' or scenario == 'abyss_cov':
    input_files = [f for f in os.listdir(path_input_maf_prob) if not f.endswith("metric_plots")]
else:
    input_files= [f for f in os.listdir(path_input_geno) if not f.startswith("mapping")]
input_files = [f for f in input_files if not f.startswith("file")]

In [ ]:
ids

In [ ]:
total

In [ ]:
path_geno_file

In [ ]:
# Initialize lists to store results
snps = []
consts = []
effect_size_snps = []
effect_size_mafs = []
std_errs = []
ts = []
Ps = []
all_betas = []
# Loop through each file in the input MAF probability dataset
for abyss_file in input_files:
    # Construct file paths for the MAF and genotype data
    path_maf_file = path_input_maf_prob + abyss_file
    path_geno_file = path_input_geno + abyss_file

    # Load genotype data and scale it
    genos = pd.read_pickle(path_geno_file)
    ids = genos.index
    scaled_data = scaler.fit_transform(genos)
    genos = pd.DataFrame(scaled_data, columns=genos.columns)
    genos = rename_duplicates(genos)  # Handling duplicate column names
    genos['FID'] = ids
    genos.set_index('FID', inplace=True)
    # Merge genotype data with phenotype data
    total = pd.merge(genos, pheno[['FID','phenotype']], on='FID')
    
    if scenario == 'cov' or scenario == 'abyss_cov':
        covs['FID'] = ids
        if scenario == 'abyss_cov':
            n_first_columns = 5
            covs = covs.iloc[:, :n_first_columns]
            covs['FID'] = ids

        total = pd.merge(total, covs, on='FID')

    
    if scenario == 'abyss' or scenario == 'abyss_cov':
        # Load MAF probability data and rename columns
        mafprob = pd.read_pickle(path_maf_file)
        mafprob.columns = [col + '_corr' for col in mafprob.columns]
        # Merge MAF probability data with the FID column
        total = pd.merge(total, mafprob, on='FID')

    filtered_columns = [col for col in total.columns 
                    if not col.startswith('PC') 
                    and not col.endswith('_corr') 
                    and col not in ['phenotype', 'FID']]
    PCs = [col for col in total.columns
                if col.startswith('PC')]

    y = total[['phenotype']]

    for snp_id in filtered_columns:
        if scenario == 'abyss':
            snp_columns = [snp_id, f"{snp_id}_corr"]
        if scenario == 'abyss_cov':
            snp_columns = [snp_id, f"{snp_id}_corr"] + PCs
        if scenario == 'cov':
            snp_columns = [snp_id] + PCs
        if scenario == 'no_cov':
            snp_columns = [snp_id]
        
        X = total[snp_columns]
        X2 = sm.add_constant(X)
        est = sm.OLS(y, X2)
        est2 = est.fit()

        est2 = est.fit()
        # Extract coefficients (betas)
        betas = est2.params            
        try:
            consts.append(betas['const'])
        except:
            consts.append(0)
        effect_size_snps.append(betas[snp_id])
        try:
            effect_size_mafs.append(betas[f"{snp_id}_corr"])
        except:
            effect_size_mafs.append(0)

        std_errs.append(est2.bse[snp_id])
        ts.append(est2.tvalues[snp_id])
            
        P = est2.pvalues[snp_id]
        Ps.append(P)
        snps.append(snp_id)
        all_betas.append(est2.params)
    sumstat = pd.DataFrame(data={
        'snp': snps,
        'consts': consts,
        'beta_snp': effect_size_snps,
        'beta_maf': effect_size_mafs,
        'std_err': std_errs,
        'Z': ts,
        'P': Ps,
        'all_betas':all_betas
        })

    try:
        split_data = sumstat['snp'].str.split(':', expand=True)
        chromosome = split_data[0]
        position = split_data[1]
        sumstat['#CHROM'] = chromosome.astype(int)
        sumstat['POS'] = position.astype(int)
        sumstat['POS'] = sumstat['POS'].astype(int)
        sumstat['-logp'] = - np.log(sumstat['P'])
    except:
        print("there is something weird")
    
    sumstat.to_pickle(path_output+f"/bin_{bin}_sumstat.pkl")

# Try plotting the summery stats

In [ ]:
sumstat = sumstat.sort_values(by='#CHROM', ascending=True)
sumstat

In [ ]:
running_pos = 0
cumulative_pos = []
for chrom, group_df in sumstat.groupby('#CHROM'):
    cumulative_pos.append(group_df['POS'] + running_pos)
    running_pos += group_df['POS'].max()

sumstat['cumulative_pos'] = pd.concat(cumulative_pos)
sumstat['SNP number'] = sumstat.index

# Define two contrasting colors from the rocket palette
palette = sns.color_palette("rocket", as_cmap=False)
color1 = palette[3]  # First color of the palette
color2 = palette[0] # Last color of the palette, for maximum contrast

# Function to assign color based on chromosome
def assign_color(chrom_number):
    return color1 if chrom_number % 2 == 0 else color2

# Apply the function to create a color column
sumstat['color'] = sumstat['#CHROM'].apply(assign_color)

chrom_midpoints = []
for chrom, group_df in sumstat.groupby('#CHROM'):
    midpoint = group_df['cumulative_pos'].min() + (group_df['cumulative_pos'].max() - group_df['cumulative_pos'].min()) / 2
    chrom_midpoints.append(midpoint)

# Plotting
g = sns.relplot(
    data=sumstat,
    x='cumulative_pos',
    y='-logp',
    aspect=2,
    hue='color',
    palette=[color1, color2],
    legend=False,
    size=3 
)


# Set x-axis ticks and labels with reversed chromosome order
chrom_labels = np.sort(sumstat['#CHROM'].unique()) # Reversing the order of chromosomes

# Set the title and x-axis, y-axis labels
g.set(title=f"GWAS Manhattan Plot for {phenotypes_labels[pheno_name]} with {scenario_labels[scenario]}")
g.set(xticks=chrom_midpoints, xticklabels=chrom_labels)
g.set_axis_labels("Chromosome", r"$-\log_{10}(P)$")  # Set y-axis label with subscript

# Set the tick labels at a 45-degree angle
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Optional: Improve the layout
plt.tight_layout()

plt.show()

# Find accuracy

In [ ]:
max_maf = 0.5
min_maf = 0
logp_min = 0

In [ ]:
sumstat_for_PRS = sumstat[sumstat['-logp'] > logp_min]

In [ ]:
sumstat_for_PRS

In [ ]:
path_input_geno = glob.PATH_GENO + f"/04_LD_blocks/{PCs}_PCs/bin_{bin}/"
mapping = [f for f in os.listdir(path_input_geno) if f.startswith("mapping")][0]
mapping_snp = pd.read_pickle(path_input_geno + mapping)
mapping_snp.rename(columns={'Position': 'POS'}, inplace=True)
mapping_snp['POS'] = mapping_snp['POS'].astype(int)
mapping_snp['ALT_FREQS'] = mapping_snp['ALT_FREQS'].astype(float)

In [ ]:
merged_df['bin'] = bin

In [ ]:
path_pheno = glob.PATH_PHENO + f"/{pheno_name}.pkl"
pheno = pd.read_pickle(path_pheno)

In [ ]:
def find_file_name(snp_name, mapping_df):
    try:
        return mapping_df.loc[snp_name, 'path']
    except Exception as e:
        print(e)
        return None

In [ ]:
unique_bins = list(merged_df['bin'].unique())
unique_bins

In [ ]:
unique_paths = list(merged_df['path'].unique())
unique_paths

In [ ]:
path_LDblocks = glob.PATH_GENO + f"/04_LD_blocks/{PCs}_PCs/"

In [ ]:
sumstat_for_PRS['bin'] = bin

In [ ]:
sumstat = sumstat_for_PRS
unique_bins = list(sumstat['bin'].unique())

In [ ]:
# Initialize PRS (Polygenic Risk Score) to 0
prs = 0

# Loop through unique bins
for unique_bin in unique_bins:
    # Select data from sumstat that matches the current unique bin
    temp_df = sumstat[sumstat['bin'] == unique_bin]

    # Construct the path to load mapping data
    path_mapping = path_LDblocks + f"bin_{bin}/mapping.pkl"
    mapp = pd.read_pickle(path_mapping)
    mapp.rename(columns={'Position': 'POS'}, inplace=True)
    mapp['POS'] = mapp['POS'].astype(int)
    mapp['ALT_FREQS'] = mapp['ALT_FREQS'].astype(float)

    # Merge data from temp_df and mapping data based on 'POS'
    merged_df = pd.merge(temp_df, mapp[['POS','ALT_FREQS','REF','ALT','path']], on='POS', how='inner')

    # Extract unique paths from merged data
    unique_paths = list(merged_df['path'].unique())

    # Loop through unique paths
    for unique_path in unique_paths:
        # Find the file with SNP information for the current unique path
        file_with_snp = [f for f in os.listdir(path_LDblocks + f"bin_{unique_bin}/") if f.startswith(unique_path)][0]
        path_file = path_LDblocks + f"bin_{unique_bin}/" + file_with_snp

        # Load genetic data from the path file
        geno = pd.read_pickle(path_file)

        # Select data from merged_df that matches the current unique path
        temp2_df = merged_df[merged_df['path'] == unique_path]

        # Iterate over rows in temp2_df
        for index, row in temp2_df.iterrows():
            try:
                # Select the genetic data for the current SNP
                genos = geno[[row['snp']]]

                # Scale the genetic data
                scaled_data = scaler.fit_transform(genos)
                genos = pd.DataFrame(scaled_data, columns=genos.columns)

                # Calculate PRS for the current SNP and add it to the overall PRS
                tmp_prs = genos * row.beta_snp
                if not tmp_prs.isna().any().any():
                    prs = prs + tmp_prs.iloc[:, 0]
            except KeyError:
                print(f"SNP {row['snp']} not found in {path_file}")
            except FileNotFoundError:
                print(f"File not found: {path_file}")


# Experimental stuff

In [ ]:
path_input_geno = glob.PATH_GENO + f"/04_LD_blocks/{PCs}_PCs/bin_{bin}/"
mapping = [f for f in os.listdir(path_input_geno) if f.startswith("mapping")][0]

In [ ]:
mapping_snp = pd.read_pickle(path_input_geno + mapping)

In [ ]:
mapping_snp.rename(columns={'Position': 'POS'}, inplace=True)
mapping_snp['POS'] = mapping_snp['POS'].astype(int)
mapping_snp['ALT_FREQS'] = mapping_snp['ALT_FREQS'].astype(float)
mapping_snp

In [ ]:
merged_df = pd.merge(sumstat, mapping_snp[['POS','ALT_FREQS','REF','ALT']], on='POS', how='inner')
merged_df

In [ ]:
running_pos = 0
cumulative_pos = []
for chrom, group_df in merged_df.groupby('#CHROM'):
    cumulative_pos.append(group_df['POS'] + running_pos)
    running_pos += group_df['POS'].max()

merged_df['cumulative_pos'] = pd.concat(cumulative_pos)
merged_df['SNP number'] = merged_df.index

# Plotting
g = sns.relplot(
    data=merged_df,
    x='cumulative_pos',
    y='-logp',
    aspect=2,
    hue='ALT_FREQS',
    palette='rocket',
    size=2,
    legend='brief' 
)

# Optional: Improve the layout
plt.tight_layout()

plt.show()

In [ ]:
from scipy.fft import ifft

# Example DataFrame with complex frequency data
data = {
    'frequency_data': [complex(1, 2), complex(3, 4), complex(5, 6)]
}
df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
merged_df[["POS",'-logp']]

In [ ]:
from scipy.interpolate import interp1d

In [ ]:
# Assuming 'df' is your DataFrame
df = pd.DataFrame({
    'freq': [957640, 190781610, 190805564, 190896859, 190932270, 36577942, 33099675, 32135488, 20278529, 50868240],
    'amp': [0.368548, 2.487326, 2.208208, 2.304231, 3.049233, 2.058717, 0.926723, 0.314953, 0.110253, 1.342221]
})

# Interpolate your data
# Create a regular grid of frequencies - adjust 'num_points' as needed
num_points = 1024  # For example
regular_freq = np.linspace(df['freq'].min(), df['freq'].max(), num_points)

# Interpolate amplitude onto this regular grid
interpolator = interp1d(df['freq'], df['amp'], kind='linear', fill_value="extrapolate")
regular_amp = interpolator(regular_freq)

# Perform Inverse Fourier Transform
time_data = ifft(regular_amp)

# Handling the result - typically, you're interested in the real part
real_part = np.real(time_data)

# Create a result DataFrame
result_df = pd.DataFrame({
    'time': np.arange(len(real_part)),
    'value': real_part
})

print(result_df)

In [ ]:
merged_df['complex'] = merged_df['-logp'].apply(lambda x: complex(0, x))

print(merged_df)

In [ ]:
test = merged_df[merged_df['ALT_FREQS'] == "3"]

In [ ]:
test

In [ ]:
merged_df['ALT_FREQS'].max()